# Notebook 24 — Final Submission Validation and Runtime Smoke Test

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 23 created the packaged submission archive.

Notebook 24 validates the archive as a standalone artifact before any Kaggle upload.

## Objectives

1. Locate the final submission ZIP.
2. Extract it into a clean temporary directory.
3. Verify the manifest and checksums.
4. Confirm the 60-card deck.
5. Compile every packaged Python file.
6. Inspect imports and runtime dependencies.
7. Load the packaged policy and agent exports.
8. Validate the packaged production entry point.
9. Test deck-request behavior.
10. Test action-selection behavior.
11. Run repeated smoke tests.
12. Detect missing, duplicate, or unexpected files.
13. Produce a final readiness report.
14. Save Notebook 24 through PowerShell.

# Cell 2 — Imports

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import tempfile
import types
import uuid
import zipfile

from pathlib import Path
from typing import Any

print("Python:", sys.version)
print("Working directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


# Cell 3 — Locate the project and archive

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = {
        "notebooks",
        "scripts",
        "src",
        "submission_work",
    }

    for candidate in [current, *current.parents]:
        found = {
            marker
            for marker in markers
            if (candidate / marker).exists()
        }

        if len(found) >= 3:
            return candidate

    return current


PROJECT_ROOT = find_project_root()

ZIP_FILE = (
    PROJECT_ROOT
    / "submission_work"
    / "team_jesus_submission.zip"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook24"
)

EXTRACT_DIR = (
    PROJECT_ROOT
    / "submission_work"
    / "notebook24_extracted"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Submission ZIP:", ZIP_FILE)
print("Extraction directory:", EXTRACT_DIR)
print("Reports:", REPORT_DIR)


Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Submission ZIP: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_submission.zip
Extraction directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\notebook24_extracted
Reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook24


# Cell 4 — Validate the ZIP exists

In [6]:
if not ZIP_FILE.is_file():
    raise FileNotFoundError(
        "Final submission ZIP was not found:\n"
        f"{ZIP_FILE}"
    )

print("Submission ZIP located.")
print("Size:", ZIP_FILE.stat().st_size, "bytes")

Submission ZIP located.
Size: 13626 bytes


# Cell 5 — Extract the submission archive

In [7]:
with zipfile.ZipFile(ZIP_FILE) as archive:
    archive.extractall(EXTRACT_DIR)
    archive_names = archive.namelist()

print("Archive entries:", len(archive_names))

for name in archive_names:
    print("-", name)

print("\nArchive extracted successfully.")

Archive entries: 9
- agent/
- data/
- evaluation/
- policy/
- manifest.json
- agent/notebook21_export.py
- data/deck.csv
- evaluation/notebook22_export.py
- policy/notebook20_export.py

Archive extracted successfully.


# Cell 6 — Define required packaged files

In [8]:
REQUIRED_PACKAGE_FILES = {
    "manifest": EXTRACT_DIR / "manifest.json",
    "deck": EXTRACT_DIR / "data" / "deck.csv",
    "agent": (
        EXTRACT_DIR
        / "agent"
        / "notebook21_export.py"
    ),
    "evaluation": (
        EXTRACT_DIR
        / "evaluation"
        / "notebook22_export.py"
    ),
    "policy": (
        EXTRACT_DIR
        / "policy"
        / "notebook20_export.py"
    ),
}

missing_files = []

for name, path in REQUIRED_PACKAGE_FILES.items():
    exists = path.is_file()

    print(
        f"{'[FOUND]' if exists else '[MISSING]'} "
        f"{name}: {path}"
    )

    if not exists:
        missing_files.append(str(path))

if missing_files:
    raise FileNotFoundError(
        "Packaged submission is incomplete:\n"
        + "\n".join(missing_files)
    )

print("\nAll required packaged files located.")

[FOUND] manifest: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\notebook24_extracted\manifest.json
[FOUND] deck: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\notebook24_extracted\data\deck.csv
[FOUND] agent: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\notebook24_extracted\agent\notebook21_export.py
[FOUND] evaluation: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\notebook24_extracted\evaluation\notebook22_export.py
[FOUND] policy: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\notebook24_extracted\policy\notebook20_export.py

All required packaged files located.


# Cell 7 — Validate the Manifest

In [9]:
import json

MANIFEST_PATH = REQUIRED_PACKAGE_FILES["manifest"]

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("Project:", manifest["project"])
print("Team:", manifest["team"])
print("Repository cards:", manifest["repository_cards"])
print("Official cards:", manifest["official_cards"])
print("Deck size:", manifest["deck_size"])

assert manifest["project"] == "PTCG AI Battle Challenge"
assert manifest["team"] == "Team Jesus"
assert manifest["repository_cards"] == 1267
assert manifest["official_cards"] == 1267
assert manifest["deck_size"] == 60

print("\nManifest validated.")

Project: PTCG AI Battle Challenge
Team: Team Jesus
Repository cards: 1267
Official cards: 1267
Deck size: 60

Manifest validated.


# Cell 8 — Verify SHA256 checksums

In [11]:
# Cell 8 — Verify SHA256 checksums

def sha256_file(path: Path) -> str:
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


checksum_results = {}

for logical_name, details in manifest["files"].items():
    relative_path = details["relative_path"]

    packaged_path = (
        EXTRACT_DIR
        / Path(relative_path)
    )

    expected_checksum = details["sha256"]
    actual_checksum = sha256_file(packaged_path)

    passed = actual_checksum == expected_checksum

    checksum_results[logical_name] = passed

    print(
        f"{'[OK]' if passed else '[FAIL]'} "
        f"{logical_name}"
    )
    print(" expected:", expected_checksum)
    print(" actual:  ", actual_checksum)
    print()

assert all(checksum_results.values())

print("All packaged checksums validated.")

[OK] agent_export
 expected: 455a6fc88760cbe638e48806673dde179007c6bc0242d66f02a6c415e05a1709
 actual:   455a6fc88760cbe638e48806673dde179007c6bc0242d66f02a6c415e05a1709

[OK] evaluation_export
 expected: b8eee4d428b38710930f52e5f242441e1491c101e366fc66e2e9096398170795
 actual:   b8eee4d428b38710930f52e5f242441e1491c101e366fc66e2e9096398170795

[OK] policy_export
 expected: cc96a7448df796f56c9f0b6d8ab6b5dd1f54ae18fcd47aadd3a636506d4b490c
 actual:   cc96a7448df796f56c9f0b6d8ab6b5dd1f54ae18fcd47aadd3a636506d4b490c

[OK] deck
 expected: b4464eb525a25e6598a972d00efc5e5b5156372e77f51853f4076d8ebb34fd7d
 actual:   b4464eb525a25e6598a972d00efc5e5b5156372e77f51853f4076d8ebb34fd7d

All packaged checksums validated.


# Cell 9 — Compile every packaged Python file

In [12]:
python_files = sorted(
    EXTRACT_DIR.rglob("*.py")
)

print("Python files found:", len(python_files))

compile_results = {}

for python_file in python_files:
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "py_compile",
            str(python_file),
        ],
        capture_output=True,
        text=True,
    )

    passed = completed.returncode == 0
    compile_results[str(python_file)] = passed

    print(
        f"{'[OK]' if passed else '[FAIL]'} "
        f"{python_file.relative_to(EXTRACT_DIR)}"
    )

    if not passed:
        print(completed.stderr)

assert python_files
assert all(compile_results.values())

print("\nAll packaged Python files compiled successfully.")

Python files found: 3
[OK] agent\notebook21_export.py
[OK] evaluation\notebook22_export.py
[OK] policy\notebook20_export.py

All packaged Python files compiled successfully.


# Cell 10 — Validate deck contents

In [13]:
deck_path = REQUIRED_PACKAGE_FILES["deck"]

deck_ids = tuple(
    int(line.strip())
    for line in deck_path.read_text(
        encoding="utf-8-sig"
    ).splitlines()
    if line.strip()
)

print("Deck size:", len(deck_ids))
print("Unique card IDs:", len(set(deck_ids)))
print("First 10 cards:", deck_ids[:10])

assert len(deck_ids) == 60
assert len(set(deck_ids)) > 1
assert all(card_id > 0 for card_id in deck_ids)

print("\nPackaged deck validated.")

Deck size: 60
Unique card IDs: 17
First 10 cards: (673, 673, 674, 674, 675, 675, 676, 676, 676, 677)

Packaged deck validated.


# Cell 11 — Check for unexpected files

In [15]:
expected_files = {
    "manifest.json",
    "data/deck.csv",
    "agent/notebook21_export.py",
    "evaluation/notebook22_export.py",
    "policy/notebook20_export.py",
}

actual_files = {
    str(path.relative_to(EXTRACT_DIR)).replace("\\", "/")
    for path in EXTRACT_DIR.rglob("*")
    if path.is_file()
}

# Ignore Python cache files
actual_files = {
    f
    for f in actual_files
    if "__pycache__" not in f
    and not f.endswith(".pyc")
}

unexpected_files = actual_files - expected_files
missing_expected = expected_files - actual_files

print("Actual files:")

for name in sorted(actual_files):
    print("-", name)

print()
print("Unexpected files:", sorted(unexpected_files))
print("Missing expected files:", sorted(missing_expected))

assert not unexpected_files
assert not missing_expected

print("\nPackage file inventory validated.")

Actual files:
- agent/notebook21_export.py
- data/deck.csv
- evaluation/notebook22_export.py
- manifest.json
- policy/notebook20_export.py

Unexpected files: []
Missing expected files: []

Package file inventory validated.


# Cell 12 - Load the packaged agent report

In [16]:
AGENT_EXPORT = REQUIRED_PACKAGE_FILES["agent"]

source_21 = AGENT_EXPORT.read_text(
    encoding="utf-8-sig"
)

cleaned_lines = [
    line
    for line in source_21.splitlines()
    if line.strip() != "from __future__ import annotations"
]

cleaned_source = (
    "from __future__ import annotations\n"
    + "\n".join(cleaned_lines)
)

module_name = (
    "packaged_notebook21_"
    + uuid.uuid4().hex
)

packaged_agent_module = types.ModuleType(
    module_name
)

packaged_agent_module.__file__ = str(
    AGENT_EXPORT
)

packaged_agent_module.__package__ = ""

sys.modules[module_name] = packaged_agent_module

compiled_agent = compile(
    cleaned_source,
    str(AGENT_EXPORT),
    "exec",
)

exec(
    compiled_agent,
    packaged_agent_module.__dict__,
)

print("Packaged agent module loaded.")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 20 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\20_policy_engine.py
Kaggle agent package: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent
Notebook 21 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook21
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 18 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\18_kaggle_observation_adapter.py
Notebook 19 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battl

,name,module,annotations
18,ApiResult,cg.api,"[state, error]"
1,AreaType,cg.api,[]
21,Attack,cg.api,"[attackId, name, text, damage, energies]"
9,Card,cg.api,"[id, serial, playerIndex]"
20,CardData,cg.api,"[cardId, name, cardType, retreatCost, hp, weak..."
3,CardType,cg.api,[]
2,EnergyType,cg.api,[]
0,IntEnum,enum,[]
15,Log,cg.api,"[type, playerIndex, hasBasicPokemon, cardId, s..."
8,LogType,cg.api,[]


Important official classes:

- Card
- CardData
- CardType
- Log
- LogType
- Option
- OptionType
- PlayerState
- Pokemon
- SearchState
- SelectContext
- SelectData
- SelectType
- State

Card
Class: <class 'cg.api.Card'>

Annotations:
- id: <class 'int'>
- serial: <class 'int'>
- playerIndex: <class 'int'>

CardData
Class: <class 'cg.api.CardData'>

Annotations:
- cardId: <class 'int'>
- name: <class 'str'>
- cardType: <enum 'CardType'>
- retreatCost: <class 'int'>
- hp: <class 'int'>
- weakness: cg.api.EnergyType | None
- resistance: cg.api.EnergyType | None
- energyType: <enum 'EnergyType'>
- basic: <class 'bool'>
- stage1: <class 'bool'>
- stage2: <class 'bool'>
- ex: <class 'bool'>
- megaEx: <class 'bool'>
- tera: <class 'bool'>
- aceSpec: <class 'bool'>
- evolvesFrom: str | None
- skills: list[cg.api.Skill]
- attacks: list[int]

CardType
Class: <enum 'CardType'>

No annotations found.

Public attributes:
- BASIC_ENERGY
- ITEM
- POKEMON
- SPECIAL_ENERGY
- STADIUM
- SUPPORTER
- TOOL
-

,name,value
0,DECK,1
1,HAND,2
2,DISCARD,3
3,ACTIVE,4
4,BENCH,5
5,PRIZE,6
6,STADIUM,7
7,ENERGY,8
8,TOOL,9
9,PRE_EVOLUTION,10


OptionType


...

SelectContext


,name,value
0,MAIN,0
1,SETUP_ACTIVE_POKEMON,1
2,SETUP_BENCH_POKEMON,2
3,SWITCH,3
4,TO_ACTIVE,4
5,TO_BENCH,5
6,TO_FIELD,6
7,TO_HAND,7
8,DISCARD,8
9,TO_DECK,9


Official cg card objects: 1267
Notebook 17 repository: 1267

Official unique Card IDs: 1267
Repository unique Card IDs: 1267
Missing from repository: 0
Extra in repository: 0
Official card sample 1
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 1
cardType: 5
energyType: 1
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {G} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 2
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 2
cardType: 5
energyType: 2
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {R} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 3
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 3
cardType: 5
energyType: 3
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {W} Energy
resistance: None
retr

,rank,option_index,option_type,semantic_label,score,reasons
0,1,0,ATTACK,Attack with Mega Lucario ex,259.913,base=100.0 | attack_bonus=150.0 | active_energ...
1,2,1,END,End Turn,-25.000,base=0.0 | end_turn_penalty=-25.0


[OK] player_feature_extraction
[OK] battle_feature_extraction
[OK] action_feature_extraction
[OK] legal_action_ranking
[OK] best_option_index

Notebook 19 validation passed.
Notebook 19 loaded.
Production functions imported.
True
True
True
True
True

Notebook dependencies verified.
Repository size: 1267
Official CardData lookup size: 1267

Policy dependencies loaded.
BattlePolicy created.
Debug mode: True
BattlePolicy created successfully.
Safe BattlePolicy created successfully.
Option index: 0
Fallback used: True
Reason: Observation adaptation failed: AttributeError: 'NoneType' object has no attribute 'current'

Fallback behavior passed.
Synthetic snapshot loaded.
Turn: 3
Legal options: 2

Synthetic snapshot validation passed.
BattlePolicy Decision
Chosen option: 0
Fallback used: False
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex

Ranked actions:
 1. index=0   type=ATTACK     score=  259.913 Attack with Mega Lucario ex
 2. index=1   type=END        score=

# Cell 13 — Verify packaged agent objects

In [17]:
required_agent_objects = [
    "KaggleBattleAgent",
    "production_agent",
    "kaggle_agent",
    "deck_ids",
    "sample_snapshot",
]

missing_agent_objects = [
    name
    for name in required_agent_objects
    if not hasattr(packaged_agent_module, name)
]

for name in required_agent_objects:
    print(
        f"{'[OK]' if hasattr(packaged_agent_module, name) else '[MISSING]'} "
        f"{name}"
    )

if missing_agent_objects:
    raise AttributeError(
        "Packaged agent module is missing:\n"
        + "\n".join(missing_agent_objects)
    )

print("\nPackaged agent interface verified.")

[OK] KaggleBattleAgent
[OK] production_agent
[OK] kaggle_agent
[OK] deck_ids
[OK] sample_snapshot

Packaged agent interface verified.


# Cell 14 — Smoke-test deck mode

In [18]:
class DeckRequest:
    select = None


packaged_kaggle_agent = (
    packaged_agent_module.kaggle_agent
)

packaged_deck_response = (
    packaged_kaggle_agent.choose(
        DeckRequest()
    )
)

print(
    "Returned deck size:",
    len(packaged_deck_response),
)

assert packaged_deck_response == list(
    packaged_agent_module.deck_ids
)

assert len(packaged_deck_response) == 60

print("\nPackaged deck mode passed.")

Returned deck size: 60

Packaged deck mode passed.


# Cell 15 — Smoke-test action mode

In [20]:
packaged_snapshot = (
    packaged_agent_module.sample_snapshot
)

packaged_action = (
    packaged_kaggle_agent.choose(
        packaged_snapshot
    )
)

print("Returned action:", packaged_action)

assert packaged_action == [0]

print("\nPackaged action mode passed.")

Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Returned action: [0]

Packaged action mode passed.


# Cell 16 — Final Validation Summary

In [25]:
# Cell 16 — Final validation summary

checksum_count = sum(
    1
    for details in manifest["files"].values()
    if details.get("sha256")
)

python_files = sorted(
    path
    for path in EXTRACT_DIR.rglob("*.py")
    if path.is_file()
)

print("=" * 72)
print("Notebook 24 — Submission Verification")
print("=" * 72)
print()

print(
    "Repository cards:",
    manifest["repository_cards"],
)
print(
    "Official cards:",
    manifest["official_cards"],
)
print(
    "Deck size:",
    manifest["deck_size"],
)

print()

print(
    "Package files:",
    len(manifest["files"]),
)
print(
    "Checksums:",
    checksum_count,
)
print(
    "Python modules:",
    len(python_files),
)

print()

print("Packaged deck: PASS")
print("Packaged action: PASS")
print("Archive integrity: PASS")
print("Manifest validation: PASS")
print("Checksum validation: PASS")

print()

print("Submission ZIP:", ZIP_FILE.name)

assert manifest["repository_cards"] == 1267
assert manifest["official_cards"] == 1267
assert manifest["deck_size"] == 60
assert len(manifest["files"]) == 4
assert checksum_count == 4
assert len(python_files) == 3
assert len(packaged_deck_response) == 60
assert packaged_action == [0]

print()
print("NOTEBOOK 24 COMPLETED SUCCESSFULLY")
print("Submission package fully verified.")

Notebook 24 — Submission Verification

Repository cards: 1267
Official cards: 1267
Deck size: 60

Package files: 4
Checksums: 4
Python modules: 3

Packaged deck: PASS
Packaged action: PASS
Archive integrity: PASS
Manifest validation: PASS
Checksum validation: PASS

Submission ZIP: team_jesus_submission.zip

NOTEBOOK 24 COMPLETED SUCCESSFULLY
Submission package fully verified.
